# Step 3: Tools & Agents in LangChain (Modern Approach) — Google Colab Version

This notebook covers the **current, non-deprecated** way to build a tool-using agent in LangChain:

1. What an agent is and why tools matter
2. Defining tools with the `@tool` decorator
3. Building an agent with `create_tool_calling_agent` + `AgentExecutor`

We'll use **Groq's `llama-3.3-70b-versatile`**, with the API key loaded from **Colab Secrets** (click the 🔑 key icon in the left sidebar, add a secret named `GROQ_API_KEY` with your key as the value, and toggle notebook access on).

Source reference: [Introduction to LangChain Agents — Aurelio AI](https://www.aurelio.ai/learn/langchain-agents-intro)

> Deprecated agent types (ReAct string-parsing agents, `initialize_agent`, etc.) are intentionally skipped.


## 1. What is an Agent, and Why Do We Need Tools?

A plain LLM can only generate text based on what it already knows — it can't check the current weather, do precise math, or look something up live. A **tool** is just a regular Python function wrapped in a way the LLM can understand: it gets a name, a description, and a defined input schema.

An **agent** is an LLM that has been given a list of tools. Instead of immediately answering, the agent can decide: "Do I need to call a tool to answer this properly, and if so, which one and with what arguments?" The LLM doesn't execute the tool itself — it just outputs *which* tool to call and *what arguments* to use. Something else (the `AgentExecutor`) actually runs the function and feeds the result back to the LLM so it can form a final answer.

So the loop is:
1. User asks a question
2. LLM decides: answer directly, or call a tool?
3. If a tool is called, the executor runs it and returns the result to the LLM
4. The LLM uses that result to either call another tool or give a final answer


## 2. Setup (Colab): Install Packages

Colab doesn't come with these packages pre-installed. Run this cell first — Colab may prompt you to restart the runtime afterward, which is fine.


In [ ]:
# Install dependencies (Colab-specific: nothing is pre-installed here)
# Pinned to versions known to work together, and where `langchain.memory`
# (ConversationBufferMemory) still exists - newer langchain 1.0+ removed it.
%pip install -qU "langchain==0.3.27" "langchain-community==0.3.27" "langchain-groq==0.2.5" "langchain-core==0.3.78"


> ⚠️ **After running the install cell above, restart the runtime**: `Runtime` → `Restart session`, then run all cells from the top again. This ensures Colab picks up the pinned versions instead of whatever was previously loaded.

In [ ]:
# Quick sanity check - confirms the pinned versions are active and langchain.memory exists
import langchain
print("langchain version:", langchain.__version__)

from langchain.memory import ConversationBufferMemory  # will raise here if pinning failed
print("langchain.memory import: OK")


## 3. Setup (Colab): Load the Groq API Key from Colab Secrets

Instead of a `.env` file (which doesn't persist in Colab), we use **Colab Secrets**:

1. Click the 🔑 key icon in the left sidebar
2. Add a new secret named `GROQ_API_KEY` with your Groq API key as the value
3. Toggle "Notebook access" on for this notebook

Then we read it with `google.colab.userdata`.


In [ ]:
import os
from google.colab import userdata
from langchain_groq import ChatGroq

# Pull the key from Colab's encrypted Secrets manager instead of a .env file
groq_api_key = userdata.get("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = groq_api_key

# Initialize the Groq-backed chat model
# temperature=0.0 keeps tool-calling decisions consistent and less random
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.0,
)

print("LLM ready:", llm.model_name if hasattr(llm, "model_name") else llm)


> **Running locally instead of Colab?** Swap the cell above for this version, which reads from a `.env` file using `python-dotenv`:
>
> ```python
> import os
> from dotenv import load_dotenv
> from langchain_groq import ChatGroq
>
> load_dotenv()
> groq_api_key = os.getenv("GROQ_API_KEY")
> if not groq_api_key:
>     raise ValueError("GROQ_API_KEY not found. Add it to your .env file.")
>
> llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.0)
> ```


## 4. Defining Tools with the `@tool` Decorator

The `@tool` decorator turns an ordinary Python function into a `StructuredTool` the agent can call. For best results, every tool should have:

- A **docstring** explaining what it does (the LLM reads this to decide when to use it)
- **Type-annotated parameters** with clear names
- A **return type annotation**

Below we define two simple tools to demonstrate multi-tool usage: a **calculator** tool and a **weather** tool.


In [ ]:
from langchain_core.tools import tool

# --- Tool 1: Calculator ---
@tool
def calculator(x: float, y: float, operation: str) -> float:
    """Perform a basic arithmetic operation on two numbers.

    operation must be one of: 'add', 'subtract', 'multiply', 'divide'.
    """
    if operation == "add":
        return x + y
    elif operation == "subtract":
        return x - y
    elif operation == "multiply":
        return x * y
    elif operation == "divide":
        if y == 0:
            raise ValueError("Cannot divide by zero.")
        return x / y
    else:
        raise ValueError(f"Unknown operation: {operation}")


# --- Tool 2: Weather (mocked, no external API key needed) ---
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city name.

    This is a mock implementation that returns fake but plausible data,
    so you can test the agent without needing a weather API key.
    """
    # In a real project, you'd call a weather API (e.g. OpenWeatherMap) here instead
    fake_weather_db = {
        "faisalabad": "32°C, sunny, light breeze",
        "islamabad": "27°C, partly cloudy",
        "lahore": "34°C, hot and humid",
    }
    key = city.strip().lower()
    if key in fake_weather_db:
        return f"The weather in {city} is currently {fake_weather_db[key]}."
    return f"No weather data found for '{city}'. (This is mock data — try Faisalabad, Islamabad, or Lahore.)"


In [ ]:
# Inspect a tool: name, description, and the schema the LLM sees
print("Name:", calculator.name)
print("Description:", calculator.description)
print("Args schema:", calculator.args_schema.model_json_schema())


## 5. Building the Agent: `create_tool_calling_agent` + `AgentExecutor`

This is the modern, recommended way to build a tool-using agent (replacing the older ReAct-style string-parsing agents and `initialize_agent`).

There are three pieces:

1. **Prompt** — a `ChatPromptTemplate` with a system message, a chat history placeholder, the user input, and an `agent_scratchpad` placeholder (where the agent tracks its intermediate tool-calling steps).
2. **Agent** — `create_tool_calling_agent(llm, tools, prompt)` wires the LLM, tools, and prompt together. Calling `agent.invoke(...)` directly only produces *one* decision (e.g. "call this tool with these args") — it does **not** actually run the tool.
3. **AgentExecutor** — `AgentExecutor(agent=agent, tools=tools, ...)` runs the full loop: call the LLM → run any requested tool → feed the result back to the LLM → repeat until a final answer is produced.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# System message + history placeholder + user input + scratchpad for tool-call reasoning
prompt = ChatPromptTemplate.from_messages([
    ("system", "You're a helpful assistant with access to tools. Use them when needed."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])


In [ ]:
from langchain.memory import ConversationBufferMemory

# Simple in-memory conversation buffer so the agent remembers earlier turns
# memory_key must match the MessagesPlaceholder variable_name above
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
)


In [ ]:
from langchain.agents import create_tool_calling_agent, AgentExecutor

# All tools available to the agent (multi-tool setup)
tools = [calculator, get_weather]

# Wires the LLM + tools + prompt together (this alone does NOT execute tools)
agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt,
)


### 5a. `agent.invoke()` Alone — Just One Decision, No Execution

Before building the `AgentExecutor`, let's see what `agent.invoke()` does by itself. It will decide *which* tool to call and *what arguments* to use, but it will **not** actually run the tool function. Notice the output is a tool-call decision object, not a final answer — that's the whole point of needing an executor next.


In [ ]:
# Calling the agent directly - it only produces a decision, it does not run the tool
agent_decision = agent.invoke({
    "input": "What is 245 multiplied by 3.5?",
    "chat_history": [],
    "intermediate_steps": [],  # agent.invoke (unlike AgentExecutor) requires this manually
})

print(agent_decision)
# Notice: this is a tool-call action (tool name + tool_input), NOT a final numeric answer.
# The multiply never actually happened yet - nothing executed calculator().


### 5b. `AgentExecutor` — Now It Actually Runs the Loop

The executor takes the same `agent` and repeatedly: asks the LLM what to do → runs the requested tool → feeds the result back → asks again → until it gets a final answer instead of another tool call.


In [ ]:
# The executor actually runs the loop: decide -> call tool -> observe -> decide again -> ...
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=memory,
    verbose=True,           # prints each step so you can see tool calls happening
    max_iterations=5,       # safety cap - stops the agent if it loops too many times
    handle_parsing_errors=True,  # if the LLM produces a malformed tool call, feed the error
                                  # back to it instead of crashing the whole run
)


## 6. Trying It Out: Multi-Tool Usage

Let's ask a question that needs the calculator, then one that needs the weather tool, and finally a question that needs **both** in a single turn — to confirm the agent can pick the right tool(s) on its own.


In [ ]:
# Should trigger the calculator tool
response = agent_executor.invoke({
    "input": "What is 245 multiplied by 3.5?",
    "chat_history": memory.chat_memory.messages,
})
print(response["output"])


In [ ]:
# Should trigger the weather tool
response = agent_executor.invoke({
    "input": "What's the weather like in Faisalabad right now?",
    "chat_history": memory.chat_memory.messages,
})
print(response["output"])


In [ ]:
# Should trigger BOTH tools in one go - tests multi-tool reasoning
response = agent_executor.invoke({
    "input": "What's the weather in Lahore, and also what is 12 plus 30 divided by 5?",
    "chat_history": memory.chat_memory.messages,
})
print(response["output"])


In [ ]:
# Memory check - does it still remember context across turns?
response = agent_executor.invoke({
    "input": "Of the two cities I asked about, which one did I ask about first?",
    "chat_history": memory.chat_memory.messages,
})
print(response["output"])


## 7. A Real Tool with Error Handling

So far our tools are either pure math or mocked data. Real tools call external services, which can fail (network errors, bad input, rate limits). Below is a **real** API call (no API key needed — uses the free [Frankfurter](https://www.frankfurter.app/) currency exchange API) with proper `try/except` handling, so the agent gets a useful error message back instead of the whole run crashing.


In [ ]:
import requests

@tool
def get_exchange_rate(base_currency: str, target_currency: str) -> str:
    """Get the current exchange rate from one currency to another.

    Use 3-letter currency codes, e.g. 'USD', 'PKR', 'EUR', 'GBP'.
    """
    base_currency = base_currency.strip().upper()
    target_currency = target_currency.strip().upper()

    try:
        response = requests.get(
            "https://api.frankfurter.app/latest",
            params={"from": base_currency, "to": target_currency},
            timeout=5,  # avoid hanging forever if the API is slow/unreachable
        )
        response.raise_for_status()  # raises an exception for 4xx/5xx responses
        data = response.json()

        rate = data.get("rates", {}).get(target_currency)
        if rate is None:
            # Handles cases like an invalid currency code that the API silently ignores
            return f"Could not find a rate for {base_currency} to {target_currency}. Check the currency codes."

        return f"1 {base_currency} = {rate} {target_currency}"

    except requests.exceptions.Timeout:
        return "The exchange rate service timed out. Please try again."
    except requests.exceptions.RequestException as e:
        # Catches connection errors, bad status codes, etc.
        return f"Error fetching exchange rate: {e}"


In [ ]:
# Add the new tool, rebuild agent + executor, and test it - including a deliberately bad input
tools = [calculator, get_weather, get_exchange_rate]

agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=memory,
    verbose=True,
    max_iterations=5,
    handle_parsing_errors=True,
)

# Normal case
response = agent_executor.invoke({
    "input": "What is the exchange rate from USD to PKR?",
    "chat_history": memory.chat_memory.messages,
})
print(response["output"])


In [ ]:
# Deliberately bad input - an invalid currency code - to see the error handling in action
response = agent_executor.invoke({
    "input": "What is the exchange rate from ZZZ to PKR?",
    "chat_history": memory.chat_memory.messages,
})
print(response["output"])
# The tool catches the bad response and returns a clear message instead of crashing -
# the agent then explains this to you in natural language.


## 8. Practice Exercise: Build Your Own Tool

Now it's your turn. Define a new tool below using the `@tool` decorator, add it to the `tools` list, rebuild the agent and executor, and test it with a question that should trigger your new tool.

Ideas to try:
- A tool that checks if a number is prime
- A tool that converts units (e.g. km to miles)
- A tool that reverses or counts characters in a string

Fill in the `TODO` sections.


In [ ]:
# TODO: Define your own tool here
@tool
def your_tool_name(param: str) -> str:
    """TODO: Describe what this tool does and when the agent should use it."""
    # TODO: implement your logic
    return "TODO: replace with real output"


# TODO: Add your tool to the tools list and rebuild the agent + executor
practice_tools = [calculator, get_weather, get_exchange_rate, your_tool_name]

practice_agent = create_tool_calling_agent(
    llm=llm,
    tools=practice_tools,
    prompt=prompt,
)

practice_executor = AgentExecutor(
    agent=practice_agent,
    tools=practice_tools,
    memory=memory,
    verbose=True,
    max_iterations=5,
    handle_parsing_errors=True,
)

# TODO: Ask a question that should trigger your new tool
result = practice_executor.invoke({
    "input": "TODO: write a question here that uses your tool",
    "chat_history": memory.chat_memory.messages,
})
print(result["output"])
